# Phase II: Exploratory Data Analysis (EDA) — Resumes (Advanced)

This notebook covers the standard EDA and additional advanced insights to better understand the resume dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer
import os
import sys

%matplotlib inline
sns.set_style("whitegrid")

## 1. Load Dataset

In [ ]:
RESUME_PATH = "../data/raw/resumes/Resume/Resume.csv"
df = pd.read_csv(RESUME_PATH)

print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Basic Distribution
Check category balance.

In [ ]:
plt.figure(figsize=(10, 8))
sns.countplot(y='Category', data=df, order=df['Category'].value_counts().index, palette='viridis')
plt.title("Resume Categories Distribution")
plt.show()

## 3. N-Gram Analysis
What are the most common 2-word and 3-word combinations across all resumes?

In [ ]:
def get_top_ngrams(corpus, n=None, g=2):
    vec = CountVectorizer(ngram_range=(g, g), stop_words='english').fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    return words_freq[:n]

top_bigrams = get_top_ngrams(df['Resume_str'].astype(str), n=15, g=2)
top_trigrams = get_top_ngrams(df['Resume_str'].astype(str), n=15, g=3)

fig, ax = plt.subplots(1, 2, figsize=(20, 8))
sns.barplot(x=[x[1] for x in top_bigrams], y=[x[0] for x in top_bigrams], ax=ax[0], palette='Blues_r')
ax[0].set_title("Top 15 Bigrams")
sns.barplot(x=[x[1] for x in top_trigrams], y=[x[0] for x in top_trigrams], ax=ax[1], palette='Greens_r')
ax[1].set_title("Top 15 Trigrams")
plt.tight_layout()
plt.show()

## 4. Category Similarity (Word Overlap)
How similar are different categories based on their vocabulary?

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Aggregate text by category
cat_text = df.groupby('Category')['Resume_str'].apply(lambda x: ' '.join(x.astype(str)))

tfidf = TfidfVectorizer(max_features=1000, stop_words='english')
cat_matrix = tfidf.fit_transform(cat_text)
sim_matrix = cosine_similarity(cat_matrix)

sim_df = pd.DataFrame(sim_matrix, index=cat_text.index, columns=cat_text.index)

plt.figure(figsize=(12, 10))
sns.heatmap(sim_df, cmap='coolwarm', annot=False)
plt.title("Category Terminology Similarity Matrix (TF-IDF Cosine Sim)")
plt.show()

## 5. Identifying Low-Content Resumes
Search for outliers with very low word counts which might be poor quality data.

In [ ]:
df['char_length'] = df['Resume_str'].astype(str).apply(len)
df['word_count'] = df['Resume_str'].astype(str).apply(lambda x: len(x.split()))

plt.figure(figsize=(10, 6))
sns.scatterplot(x='char_length', y='word_count', data=df, hue='Category', alpha=0.5, legend=False)
plt.title("Char Length vs Word Count Correlation")
plt.show()

print("Resumes with < 50 words:")
print(df[df['word_count'] < 50][['ID', 'Category', 'word_count']])